In [2]:
!pip install scikeras

In [4]:
!pip install bayesian-optimization

In [5]:

import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import time
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import make_scorer, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.multiclass import type_of_target
import tensorflow as tf
from numpy import unique
from numpy import reshape
from tensorflow.keras.models import Sequential
from sklearn.model_selection import cross_val_score
from tensorflow.keras.layers import Input, Conv1D, Dense, Dropout, BatchNormalization, Flatten, MaxPooling1D
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adadelta, Adagrad, Adamax, Nadam, Ftrl
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from scikeras.wrappers import KerasClassifier  # Use scikeras for scikit-learn compatibility
from math import floor
from bayes_opt import BayesianOptimization
from tensorflow.keras.layers import LeakyReLU  # Use tensorflow.keras instead of keras
LeakyReLU = LeakyReLU(negative_slope=0.1)
import warnings
# Set option to ensure charts are displayed inline in the notebook

%matplotlib inline

In [6]:
#Create a path to where your data is stored.
path = r'/Users/april/Machine Learning'

In [7]:
#Create a path to where your data is stored.
path2 = r'/Users/april/Machine Learning/Datasets'

In [11]:
# Export weather data
cleaned = pd.read_csv(os.path.join(path, 'weather_clean.csv'), index_col =False)

In [12]:
#Read in the pleasant weather data.
pleasantweather = pd.read_csv(os.path.join(path2, 'Supervised','Dataset-Answers-Weather_Prediction_Pleasant_Weather.csv'))
pleasantweather

,DATE,BASEL_pleasant_weather,BELGRADE_pleasant_weather,BUDAPEST_pleasant_weather,DEBILT_pleasant_weather,DUSSELDORF_pleasant_weather,HEATHROW_pleasant_weather,KASSEL_pleasant_weather,LJUBLJANA_pleasant_weather,MAASTRICHT_pleasant_weather,MADRID_pleasant_weather,MUNCHENB_pleasant_weather,OSLO_pleasant_weather,SONNBLICK_pleasant_weather,STOCKHOLM_pleasant_weather,VALENTIA_pleasant_weather
0,19600101,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,19600102,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,19600103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,19600104,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,19600105,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22945,20221027,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22946,20221028,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22947,20221029,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
22948,20221030,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Data Wrangling

In [13]:
# Drop DATE column from answers

pleasantweather.drop(columns = 'DATE', inplace = True)

In [14]:
pleasantweather.shape

(22950, 15)

In [15]:
cleaned.shape

(22950, 137)

In [21]:
# Turn X and answers from a df to arrays

X = np.array(cleaned)
y = np.array(pleasantweather)

In [22]:
# Use argmax to transform y

y =  np.argmax(y, axis = 1)
y

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [23]:
# Check y layout

from sklearn.utils.multiclass import type_of_target
type_of_target(y)

'multiclass'

In [24]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [25]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17212, 137) (17212,)
(5738, 137) (5738,)


## Bayesian Hyperparameter Optimization

In [49]:
# --- 1) reshape helper for CNNs (tabular -> (n, features, 1)) ---
import numpy as np
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline

def to_3d(X):
    X = np.asarray(X)
    return X.reshape(X.shape[0], X.shape[1], 1) if X.ndim == 2 else X

In [50]:
# --- 2) define a CNN that expects (timesteps, 1), NOT (timesteps, timesteps) ---
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Dropout, BatchNormalization

def cnn_model(meta, neurons=64, kernel=3, activation="relu",
              normalization=False, dropout=False, dropout_rate=0.3, n_classes=None):
    timesteps = meta["n_features_in_"]          # e.g., 137
    if n_classes is None:
        # safer default; scikeras sets classes_ later, but we can compute on fit data
        n_classes = len(np.unique(meta["y"].ravel()))
    m = Sequential([
        Input(shape=(timesteps, 1)),           # <<<< THIS is the key change
        Conv1D(int(neurons), kernel_size=int(max(1, kernel)), activation=activation),
        Conv1D(int(neurons), kernel_size=int(max(1, kernel)), activation=activation),
        BatchNormalization() if normalization else Input(shape=()) ,  # no-op if False
        GlobalMaxPooling1D(),
        Dense(max(8, int(neurons//2)), activation=activation),
        Dropout(float(dropout_rate)) if dropout else Input(shape=()),
        Dense(n_classes, activation="softmax"),
    ])
    # remove no-op Input layers if inserted
    m._layers = [l for l in m.layers if not isinstance(l, Input.__class__)]
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

In [52]:
def bay_area(neurons, activation, kernel, optimizer, learning_rate,
             batch_size, epochs, layers1, layers2, normalization, dropout, dropout_rate):
    try:
        # cast/map params ...
        neurons     = int(round(neurons))
        kernel      = int(max(1, round(kernel)))
        batch_size  = int(max(8, round(batch_size)))
        epochs      = int(max(5, round(epochs)))
        activation  = "relu"  # or map from index if you’re using a list
        normalization = bool(round(normalization))
        dropout       = bool(round(dropout))
        dropout_rate  = float(np.clip(dropout_rate, 0.0, 0.7))

        clf = KerasClassifier(
            model=cnn_model,
            neurons=neurons, kernel=kernel, activation=activation,
            normalization=normalization, dropout=dropout, dropout_rate=dropout_rate,
            epochs=epochs, batch_size=batch_size, verbose=0,
        )
        nn = make_pipeline(FunctionTransformer(to_3d, validate=False), clf)

        kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        scores = cross_val_score(
            nn, X, y, scoring="accuracy", cv=kfold, n_jobs=1, error_score="raise",
            **{"fit__callbacks":[es], "fit__validation_split":0.1}
        )
        return float(np.nanmean(scores))
    except Exception as e:
        import warnings
        warnings.warn(f"BO trial failed: {e}")
        return 1e-6

In [53]:
start = time.time()
params ={
    'neurons': (10, 100),
    'kernel': (1, 3),
    'activation':(0, 9), 
    'optimizer':(0,7),
    'learning_rate':(0.01, 1),
    'batch_size': (200, 1000), 
    'epochs':(20, 50),
    'layers1':(1,3),
    'layers2':(1,3),
    'normalization':(0,1),
    'dropout':(0,1),
    'dropout_rate':(0,0.3)
}
# Run Bayesian Optimization
nn_opt = BayesianOptimization(bay_area, params, random_state=42)
nn_opt.maximize(init_points=15, n_iter=4) 
print('Search took %s minutes' % ((time.time() - start)/60))

|   iter    |  target   |  neurons  |  kernel   | activa... | optimizer | learni... | batch_... |  epochs   |  layers1  |  layers2  | normal... |  dropout  | dropou... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | 1e-06     | 43.708610 | 2.9014286 | 6.5879454 | 4.1906093 | 0.1644584 | 324.79561 | 21.742508 | 2.7323522 | 2.2022300 | 0.7080725 | 0.0205844 | 0.2909729 |
| 2         | 1e-06     | 84.919837 | 1.4246782 | 1.6364247 | 1.2838315 | 0.3111998 | 619.80514 | 32.958350 | 1.5824582 | 2.2237057 | 0.1394938 | 0.2921446 | 0.1099085 |
| 3         | 1e-06     | 51.046298 | 2.5703519 | 1.7970640 | 3.5996410 | 0.5964904 | 237.16033 | 38.226345 | 1.3410482 | 1.1301031 | 0.9488855 | 0.9656320 | 0.2425192 |
| 4         | 1e-06     | 37.415239 | 1.1953442 | 6.1580972 | 3.0810674 | 0.1308178 | 596.14152 | 21.031655 | 2.8186408 | 1.5175599 | 0.6625222 | 0.31

C:\Users\april\AppData\Local\Temp\ipykernel_19948\353848194.py:30: UserWarning: BO trial failed: got an unexpected keyword argument 'fit__callbacks'
  warnings.warn(f"BO trial failed: {e}")
C:\Users\april\AppData\Local\Temp\ipykernel_19948\353848194.py:30: UserWarning: BO trial failed: got an unexpected keyword argument 'fit__callbacks'
  warnings.warn(f"BO trial failed: {e}")
C:\Users\april\AppData\Local\Temp\ipykernel_19948\353848194.py:30: UserWarning: BO trial failed: got an unexpected keyword argument 'fit__callbacks'
  warnings.warn(f"BO trial failed: {e}")


| 17        | 1e-06     | 99.979678 | 2.2730677 | 2.9022327 | 5.9409046 | 0.2304814 | 213.77910 | 23.010813 | 1.1161986 | 1.4808260 | 0.7157986 | 0.2213048 | 0.2077561 |


C:\Users\april\AppData\Local\Temp\ipykernel_19948\353848194.py:30: UserWarning: BO trial failed: got an unexpected keyword argument 'fit__callbacks'
  warnings.warn(f"BO trial failed: {e}")


| 18        | 1e-06     | 99.160714 | 2.7775362 | 0.7887248 | 1.8595826 | 0.2242737 | 995.37778 | 20.906365 | 2.0398645 | 2.1555145 | 0.5747379 | 0.4268439 | 0.2705638 |
| 19        | 1e-06     | 99.137732 | 2.8768244 | 4.5750706 | 4.3763439 | 0.0649609 | 213.21673 | 46.156765 | 2.0960735 | 2.9235409 | 0.8431643 | 0.5611792 | 0.1709171 |
Search took 0.020629175504048667 minutes


C:\Users\april\AppData\Local\Temp\ipykernel_19948\353848194.py:30: UserWarning: BO trial failed: got an unexpected keyword argument 'fit__callbacks'
  warnings.warn(f"BO trial failed: {e}")


In [54]:
optimum = nn_opt.max['params']
learning_rate = optimum['learning_rate']

activationL = ['relu', 'sigmoid', 'softplus', 'softsign', 'tanh', 'selu', 'elu', 'exponential', LeakyReLU, 'relu']
optimum['activation'] = activationL[round(optimum['activation'])]

optimum['batch_size'] = round(optimum['batch_size'])
optimum['epochs'] = round(optimum['epochs'])
optimum['layers1'] = round(optimum['layers1'])
optimum['layers2'] = round(optimum['layers2'])
optimum['neurons'] = round(optimum['neurons'])

optimizerL = ['Adam', 'SGD', 'RMSprop', 'Adadelta', 'Adagrad', 'Adamax', 'Nadam', 'Ftrl', 'Adam']
optimizerD = {
    'Adam': Adam(learning_rate=learning_rate),
    'SGD': SGD(learning_rate=learning_rate),
    'RMSprop': RMSprop(learning_rate=learning_rate),
    'Adadelta': Adadelta(learning_rate=learning_rate),
    'Adagrad': Adagrad(learning_rate=learning_rate),
    'Adamax': Adamax(learning_rate=learning_rate),
    'Nadam': Nadam(learning_rate=learning_rate),
    'Ftrl': Ftrl(learning_rate=learning_rate)
}
optimum['optimizer'] = optimizerD[optimizerL[round(optimum['optimizer'])]]
optimum

{'neurons': 44,
 'kernel': 2.9014286128198323,
 'activation': 'exponential',
 'optimizer': <keras.src.optimizers.adagrad.Adagrad at 0x18e0e9714f0>,
 'learning_rate': 0.16445845403801215,
 'batch_size': 325,
 'epochs': 22,
 'layers1': 3,
 'layers2': 2,
 'normalization': 0.7080725777960455,
 'dropout': 0.020584494295802447,
 'dropout_rate': 0.29097295564859826}

In [57]:
# Set the model with optimized hyperparameters

epochs = 47
batch_size = 460

timesteps = len(X_train[0])
input_dim = len(X_train[0])
n_classes = 15

layers1 = 1
layers2 = 2
activation = 'softsign'
kernel = int(round(1.9444298503238986))  # Rounded kernel size for Conv1D
neurons = 61
normalization = 0.770967179954561
dropout = 0.7296061783380641
dropout_rate = 0.19126724140656393
optimizer = Adadelta(learning_rate=0.7631771981307285)  # Instantiate RMSprop with learning rate

model = Sequential()
model.add(Conv1D(neurons, kernel_size=kernel, activation=activation, input_shape=(timesteps, input_dim)))

if normalization > 0.5:
    model.add(BatchNormalization())

for i in range(layers1):
    model.add(Dense(neurons, activation=activation))

if dropout > 0.5:
    model.add(Dropout(dropout_rate))

for i in range(layers2):
    model.add(Dense(neurons, activation=activation))

model.add(MaxPooling1D())
model.add(Flatten())
model.add(Dense(n_classes, activation='softmax')) 

model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

C:\Users\april\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [58]:
model.summary()

Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_20 (Conv1D)              │ (None, 136, 61)        │        16,775 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 136, 61)        │           244 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_120 (Dense)               │ (None, 136, 61)        │         3,782 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 136, 61)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_121 (Dense)               │ (None, 136, 61)        │         3,782 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_122 (Dense)               │ (None, 136, 61)        │         3,782 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_20 (MaxPooling1D) │ (None, 68, 61)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_20 (Flatten)            │ (None, 4148)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_123 (Dense)               │ (None, 15)             │        62,235 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 90,600 (353.91 KB)

 Trainable params: 90,478 (353.43 KB)

 Non-trainable params: 122 (488.00 B)

In [62]:
# Put the y_test set back into a one-hot configuration

y_train_one_hot = to_categorical(y_train, num_classes=15)

In [63]:
# Check shapes

print(f'X_train shape: {X_train.shape}')
print(f'y_train_one_hot shape: {y_train_one_hot.shape}')

X_train shape: (17212, 137)
y_train_one_hot shape: (17212, 15)


In [64]:
# Compile the model with categorical_crossentropy

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [67]:
from tensorflow.keras import Sequential, Input, backend as K

In [68]:
n_features = X_train.shape[1]
n_classes  = y_train_one_hot.shape[1]

# reshape inputs
X_train_3d = X_train.reshape(X_train.shape[0], n_features, 1)
# Do the same for X_val/X_test when you use them

K.clear_session()

model = Sequential([
    Input(shape=(n_features, 1)),        # <<< (137,1) not (137,137)
    Conv1D(64, 3, activation="relu"),
    Conv1D(64, 3, activation="relu"),
    GlobalMaxPooling1D(),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(n_classes, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="categorical_crossentropy",   # one-hot labels
              metrics=["accuracy"])

model.summary()
model.fit(X_train_3d, y_train_one_hot, batch_size=batch_size, epochs=epochs, verbose=2)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 135, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 133, 64)        │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           975 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,743 (69.31 KB)

 Trainable params: 17,743 (69.31 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/47
38/38 - 4s - 93ms/step - accuracy: 0.4062 - loss: 288995.4688
Epoch 2/47
38/38 - 2s - 50ms/step - accuracy: 0.5084 - loss: 9849.7324
Epoch 3/47
38/38 - 2s - 51ms/step - accuracy: 0.6348 - loss: 96.7229
Epoch 4/47
38/38 - 2s - 49ms/step - accuracy: 0.5999 - loss: 77.3363
Epoch 5/47
38/38 - 2s - 51ms/step - accuracy: 0.6204 - loss: 1.2663
Epoch 6/47
38/38 - 2s - 53ms/step - accuracy: 0.6258 - loss: 1.2374
Epoch 7/47
38/38 - 2s - 52ms/step - accuracy: 0.6191 - loss: 21.8252
Epoch 8/47
38/38 - 2s - 47ms/step - accuracy: 0.6247 - loss: 1.2093
Epoch 9/47
38/38 - 2s - 48ms/step - accuracy: 0.6263 - loss: 1.1931
Epoch 10/47
38/38 - 2s - 53ms/step - accuracy: 0.6295 - loss: 1.1868
Epoch 11/47
38/38 - 2s - 56ms/step - accuracy: 0.6286 - loss: 1.1753
Epoch 12/47
38/38 - 2s - 55ms/step - accuracy: 0.6349 - loss: 1.1585
Epoch 13/47
38/38 - 2s - 56ms/step - accuracy: 0.6324 - loss: 1.1546
Epoch 14/47
38/38 - 2s - 51ms/step - accuracy: 0.6351 - loss: 1.1433
Epoch 15/47
38/38 - 2s - 53ms/st

## Creating Confusion Matrix

In [69]:
# Define list of stations names

stations = {
0: 'BASEL',
1: 'BELGRADE',
2: 'BUDAPEST',
3: 'DEBILT',
4: 'DUSSELDORF',
5: 'HEATHROW',
6: 'KASSEL',
7: 'LJUBLJANA',
8: 'MAASTRICHT',
9: 'MADRID',
10: 'MUNCHENB',
11: 'OSLO',
12: 'SONNBLICK',
13: 'STOCKHOLM',
14: 'VALENTIA'
}

In [70]:
def confusion_matrix(y_true, y_pred, stations):
    # Check if y_true and y_pred are one-hot encoded or already class indices
    if y_true.ndim == 1:
        y_true_labels = y_true
    else:
        y_true_labels = np.argmax(y_true, axis=1)
    
    if y_pred.ndim == 1:
        y_pred_labels = y_pred
    else:
        y_pred_labels = np.argmax(y_pred, axis=1)
        
    # Map numeric labels to activity names
    y_true_series = pd.Series([stations[y] for y in y_true_labels])
    y_pred_series = pd.Series([stations[y] for y in y_pred_labels])
    
    return pd.crosstab(y_true_series, y_pred_series, rownames=['True'], colnames=['Pred'])

In [71]:
y_pred = model.predict(X_test)

180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [72]:
# Evaluate

print(confusion_matrix(y_test, y_pred, stations))

Pred        BASEL  BELGRADE
True                       
BASEL        3416       266
BELGRADE      669       423
BUDAPEST      139        75
DEBILT         51        31
DUSSELDORF     16        13
HEATHROW       54        28
KASSEL          6         5
LJUBLJANA      42        19
MAASTRICHT      8         1
MADRID        371        87
MUNCHENB        6         2
OSLO            5         0
STOCKHOLM       2         2
VALENTIA        1         0
